# SecBERT Fine-Tuning for Cybersecurity NER

```
jackaduma/SecBERT  (pre-trained with MLM on cybersecurity corpora by its original authors)
        ↓
Fine-tune for Cybersecurity NER (Token Classification)
        ↓
Evaluate (Precision, Recall, F1)
        ↓
Demo
```

The base model, [`jackaduma/SecBERT`](https://huggingface.co/jackaduma/SecBERT), was pre-trained with Masked Language Modeling on security-domain corpora (APTnotes, Stucco-Data, CASIE, SemEval-2018 Task 8) by its original authors. This notebook fine-tunes it for cybersecurity Named Entity Recognition on the CyberNER dataset (31-label BIO schema, 15 entity types) and evaluates it with entity-level Precision, Recall, and F1.

Requires a GPU runtime: **Runtime → Change runtime type → T4 GPU**.

In [ ]:
%pip install -q -U transformers datasets evaluate seqeval accelerate

## 1. Load dataset files from Google Drive

Reads `cyberner_clean.csv` and `ner_cyber_labels.json` from `MyDrive/datasets/ner/`.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DATA_DIR = "/content/drive/MyDrive/datasets/ner"
OUTPUT_DIR = "/content/drive/MyDrive/models"

!mkdir -p "{OUTPUT_DIR}"
!cp "{DATA_DIR}/cyberner_clean.csv" "{DATA_DIR}/ner_cyber_labels.json" .

## 2. Prepare the CyberNER dataset

Raw tags are mapped to the 31-label cyber schema; unmapped tags become `O`. Tokens are grouped into sentences by `Sentence_ID`.

In [ ]:
import json
import pandas as pd

with open("ner_cyber_labels.json", encoding="utf-8") as f:
    schema = json.load(f)

label_list = schema["label_list"]
tag_mapping = schema["tag_mapping"]
label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for l, i in label2id.items()}

df = pd.read_csv("cyberner_clean.csv")
df["Word"] = df["Word"].fillna("#")
df["Tag"] = df["Tag"].fillna("O").map(lambda t: tag_mapping.get(t, "O"))

grouped = df.groupby("Sentence_ID").agg({"Word": list, "Tag": list}).reset_index()
grouped["ner_tags"] = grouped["Tag"].apply(lambda tags: [label2id.get(t, label2id["O"]) for t in tags])
grouped = grouped.rename(columns={"Word": "tokens"})

print(f"Labels: {len(label_list)}  |  Sentences: {len(grouped)}")

## 3. Train / validation / test splits

80/20 train/test, then 80/20 of train for validation (seed 42). `scripts/evaluate_ner.py` re-derives the same test split locally.

In [ ]:
from datasets import Dataset

full_ds = Dataset.from_pandas(grouped[["tokens", "ner_tags"]])
split1 = full_ds.train_test_split(test_size=0.2, seed=42)
split2 = split1["train"].train_test_split(test_size=0.2, seed=42)
train_ds, val_ds, test_ds = split2["train"], split2["test"], split1["test"]

print(f"train={len(train_ds)}  val={len(val_ds)}  test={len(test_ds)}")

## 4. Tokenization and label alignment

Only the first sub-token of each word carries the label; remaining sub-tokens and special tokens are set to `-100` and ignored by the loss and metrics.

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "jackaduma/SecBERT"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        max_length=512,
        is_split_into_words=True,
    )
    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids)
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

tokenized_train = train_ds.map(tokenize_and_align_labels, batched=True)
tokenized_val = val_ds.map(tokenize_and_align_labels, batched=True)
tokenized_test = test_ds.map(tokenize_and_align_labels, batched=True)

## 5. Load SecBERT with a token-classification head

The warning that classifier weights are newly initialized is expected — that head is what fine-tuning trains.

In [ ]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
)

## 6. Evaluation metrics (entity-level Precision, Recall, F1)

In [ ]:
import numpy as np
import evaluate

seqeval_metric = evaluate.load("seqeval")

def compute_metrics(eval_preds):
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)
    true_predictions = [
        [id2label[p] for p, l in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]
    true_labels = [
        [id2label[l] for p, l in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]
    results = seqeval_metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

## 7. Fine-tune

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForTokenClassification

training_args = TrainingArguments(
    output_dir="secbert_ner",
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=10,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    save_total_limit=2,
    fp16=True,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=DataCollatorForTokenClassification(tokenizer),
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()

## 8. Evaluate on the held-out test set

In [ ]:
from seqeval.metrics import classification_report
from seqeval.scheme import IOB2

test_output = trainer.predict(tokenized_test)

print("Test-set entity-level metrics:")
for key in ("test_precision", "test_recall", "test_f1", "test_accuracy"):
    print(f"  {key.replace('test_', '').capitalize():<10} {test_output.metrics[key]:.4f}")

predictions = np.argmax(test_output.predictions, axis=-1)
labels = test_output.label_ids
true_predictions = [
    [id2label[p] for p, l in zip(pred, lab) if l != -100]
    for pred, lab in zip(predictions, labels)
]
true_labels = [
    [id2label[l] for p, l in zip(pred, lab) if l != -100]
    for pred, lab in zip(predictions, labels)
]

print(classification_report(true_labels, true_predictions, mode="strict", scheme=IOB2, zero_division=0))

## 9. Save the model to Google Drive (`MyDrive/models/`)

In [ ]:
trainer.save_model("secbert_ner_final")
tokenizer.save_pretrained("secbert_ner_final")

!zip -r -q secbert_ner_final.zip secbert_ner_final
!cp secbert_ner_final.zip "{OUTPUT_DIR}/"

print(f"Saved to {OUTPUT_DIR}/secbert_ner_final.zip")

## 10. Deploy locally

1. Download `secbert_ner_final.zip` from `MyDrive/models/` and extract it to `models/secbert_ner_final/` in the project root.
2. Reproduce the evaluation: `python scripts/evaluate_ner.py`
3. Run the demo: `python backend/ner_api.py`, then `cd frontend && npm start`.